In [1]:
!python make_messy_data.py


Created messy_sales.csv with 315 rows


In [2]:
import pandas as pd

df = pd.read_csv("messy_sales.csv")
print(df.shape)
df.head(10)

(315, 7)


,Order ID,Order Date,Customer Name,City,Product,Amount (Rs),Qty
0,ORD1074,23-10-2026,Rahul,Pune,bag,806,1
1,ORD1270,2026/10/22,Rahul,Pune,Headphones,"₹1,329",4
2,ORD1031,04 Jun 2026,Amit,Mumbai,bag,"4,650",1
3,ORD1038,15-07-2026,Priya,Bangalore,shoes,1085,3
4,ORD1169,19-08-2026,Rahul,Delhi,headphones,"₹1,456",4
5,ORD1071,2026/02/21,Rahul,JAIPUR,Headphones,2518,5
6,ORD1172,2026/02/14,Karan,Pune,SHOES,"3,846",5
7,ORD1089,2026/05/20,Priya,Pune,WATCH,"₹4,368",2
8,ORD1281,17 Jun 2026,Neha,Bangalore,BAG,NaN,3
9,ORD1067,28-07-2026,NaN,Mumbai,shirt,"₹2,177",1


In [3]:
df.info()
print(df.isna().sum())
print(df["City"].unique())
print("Duplicates:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order ID       315 non-null    object
 1   Order Date     315 non-null    object
 2   Customer Name  240 non-null    object
 3   City           315 non-null    object
 4   Product        315 non-null    object
 5   Amount (Rs)    306 non-null    object
 6   Qty            315 non-null    int64 
dtypes: int64(1), object(6)
memory usage: 17.4+ KB
Order ID          0
Order Date        0
Customer Name    75
City              0
Product           0
Amount (Rs)       9
Qty               0
dtype: int64
['Pune ' 'Mumbai ' ' Bangalore ' 'Delhi ' 'JAIPUR' 'Pune' ' Pune '
 ' Mumbai ' 'Bangalore' 'BANGALORE' 'Jaipur ' ' Jaipur ' 'bangalore'
 'Delhi' 'Bangalore ' 'Jaipur' 'delhi' ' Delhi ' 'mumbai' 'jaipur' 'PUNE'
 'MUMBAI' 'Mumbai' 'pune' 'DELHI']
Duplicates: 15


In [4]:
df.columns = (
    df.columns
    .str.strip()                                  # extra spaces 
    .str.lower()                                  # lower case
    .str.replace(r"[^\w\s]", "", regex=True)      # ( ) jaise symbols hatao
    .str.replace(r"\s+", "_", regex=True)         # spaces ko _ banao
)

print(df.columns.tolist())

['order_id', 'order_date', 'customer_name', 'city', 'product', 'amount_rs', 'qty']


In [5]:
df.isnull().sum()

order_id          0
order_date        0
customer_name    75
city              0
product           0
amount_rs         9
qty               0
dtype: int64

In [6]:
df=df.drop_duplicates()
print(df)

    order_id   order_date customer_name         city     product amount_rs  \
0    ORD1074   23-10-2026         Rahul        Pune          bag       806   
1    ORD1270   2026/10/22         Rahul        Pune   Headphones    ₹1,329   
2    ORD1031  04 Jun 2026          Amit      Mumbai          bag     4,650   
3    ORD1038   15-07-2026         Priya   Bangalore        shoes      1085   
4    ORD1169   19-08-2026         Rahul       Delhi   headphones    ₹1,456   
..       ...          ...           ...          ...         ...       ...   
310  ORD1212   30-08-2026         Priya   Bangalore          Bag     4,699   
311  ORD1068  27 Jul 2026         Karan       Delhi        WATCH     2,015   
312  ORD1026   2026/06/10         Rahul       Jaipur  HEADPHONES       699   
313  ORD1197  30 Sep 2026          Sana   Bangalore        shoes    ₹3,965   
314  ORD1176  26 Jun 2026         Rahul         pune         bag      4438   

     qty  
0      1  
1      4  
2      1  
3      3  
4      4

In [7]:
df["city"] = df["city"].str.strip().str.title()
df["product"] = df["product"].str.strip().str.title()
print(df["city"].unique())   # sirf 5 shehar aane chahiye

['Pune' 'Mumbai' 'Bangalore' 'Delhi' 'Jaipur']


In [8]:
df['amount_rs'].dtype

dtype('O')

In [9]:
df["amount_rs"] = (
    df["amount_rs"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["amount_rs"] = pd.to_numeric(df["amount_rs"], errors="coerce")  # "N/A" -> NaN

In [10]:
formats = ["%d-%m-%Y", "%Y/%m/%d", "%d %b %Y"]
parsed = pd.Series(pd.NaT, index=df.index)
for fmt in formats:
    parsed = parsed.fillna(pd.to_datetime(df["order_date"], format=fmt, errors="coerce"))
df["order_date"] = parsed
print(df["order_date"].isna().sum()) 


0


In [11]:
df["customer_name"] = df["customer_name"].fillna("Unknown")

In [12]:
bad_rows = df[df["order_date"].isna() | df["amount_rs"].isna()]
print("Bad rows:", len(bad_rows))
clean_df = df.dropna(subset=["order_date", "amount_rs"])

Bad rows: 9


In [13]:
by_city = (clean_df.groupby("city")
           .agg(total_sales=("amount_rs", "sum"), orders=("order_id", "count"))
           .sort_values("total_sales", ascending=False))

monthly = (clean_df.groupby(clean_df["order_date"].dt.to_period("M").astype(str))
           .agg(total_sales=("amount_rs", "sum"), orders=("order_id", "count")))

top_products = (clean_df.groupby("product")["amount_rs"].sum()
                .sort_values(ascending=False).to_frame("total_sales"))

In [14]:
with pd.ExcelWriter("sales_report.xlsx", engine="openpyxl") as w:
    clean_df.to_excel(w, sheet_name="clean_data", index=False)
    bad_rows.to_excel(w, sheet_name="bad_rows", index=False)
    by_city.to_excel(w, sheet_name="by_city")
    monthly.to_excel(w, sheet_name="monthly")
    top_products.to_excel(w, sheet_name="top_products")